# DME-Encoder Diffusion Pipeline - Kaggle Notebook

This notebook trains the A8 multi-step diffusion denoising objective, reports diffusion validation metrics, fine-tunes the pretrained encoder, and can run optional fixed-K event suffix generation.


In [ ]:
# Cell 1 - Setup & Install
import json
import pathlib
import pickle
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

WORKING_DIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
REPO_CANDIDATES = [
    WORKING_DIR / "denoising-event-sequences",
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
]
REPO_DIR = next(
    (p for p in REPO_CANDIDATES if (p / "src").exists() and (p / "scripts").exists()),
    REPO_CANDIDATES[0],
)
if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository not found. Checked: {REPO_CANDIDATES}")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--quiet"], check=True)
sys.path.insert(0, str(REPO_DIR))

required_imports = {
    "yaml": "pyyaml",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "torchmetrics": "torchmetrics",
    "matplotlib": "matplotlib",
}
missing_packages = []
for module_name, package_name in required_imports.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages, "--quiet"], check=True)

import numpy as np
import pandas as pd
import torch
import yaml

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

print(f"Repo       : {REPO_DIR}")
print(f"Device     : {device}")
print(f"Mixed prec : {USE_AMP}")
print(f"PyTorch    : {torch.__version__}")


In [ ]:
# Cell 2 - Paths & Runtime Knobs
DATA_ROOT = WORKING_DIR / "data"
PROCESSED_DIR = DATA_ROOT / "processed" / "rosbank_diffusion"
OUTPUT_DIR = WORKING_DIR / "outputs" / "diffusion"
LOG_DIR = OUTPUT_DIR / "logs"
PRETRAIN_OUTPUT_DIR = OUTPUT_DIR / "pretrain"
FROZEN_OUTPUT_DIR = OUTPUT_DIR / "finetune_frozen"
FULL_OUTPUT_DIR = OUTPUT_DIR / "finetune_full"
LOW_LABEL_OUTPUT_DIR = OUTPUT_DIR / "low_label"
GENERATION_OUTPUT_DIR = OUTPUT_DIR / "generation"
CONFIG_PATH = WORKING_DIR / "diffusion_pipeline.yaml"
RAW_PARQUET_PATH = DATA_ROOT / "rosbank_diffusion_raw.parquet"

for path in [
    DATA_ROOT,
    PROCESSED_DIR,
    OUTPUT_DIR,
    LOG_DIR,
    PRETRAIN_OUTPUT_DIR,
    FROZEN_OUTPUT_DIR,
    FULL_OUTPUT_DIR,
    LOW_LABEL_OUTPUT_DIR,
    GENERATION_OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

FAST_DEBUG = False
REBUILD_DATA = True
RUN_DIFFUSION_PRETRAIN = True
RUN_FROZEN_FINETUNE = True
RUN_FULL_FINETUNE = True
RUN_LOW_LABEL = False
RUN_EVENT_GENERATION = True

NUM_EPOCHS_PRETRAIN = 1 if FAST_DEBUG else 30
NUM_EPOCHS_FINETUNE = 1 if FAST_DEBUG else 20
BATCH_SIZE = 32 if FAST_DEBUG else 128
DIFFUSION_STEPS = 25 if FAST_DEBUG else 100
SMOKE_ENTITY_LIMIT = 64 if FAST_DEBUG else 512
SMOKE_BATCH_SIZE = 4 if FAST_DEBUG else 8
GENERATION_ENTITY_LIMIT = 8 if FAST_DEBUG else 64
GENERATION_NUM_SAMPLES = 1
GENERATION_SUFFIX_LEN = 8 if FAST_DEBUG else 16
GENERATION_SAMPLING_STEPS = 5 if FAST_DEBUG else 50
RAW_DATA_PATH = DATA_ROOT / "rosbank" / "train.csv.gz"


def run_logged(cmd: list[str], log_path: pathlib.Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")


def load_jsonl(path: pathlib.Path) -> list[dict]:
    if not path.exists():
        print(f"Metrics file is not available yet: {path}")
        return []
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def show_metrics_tail(path: pathlib.Path, title: str, tail: int = 8) -> pd.DataFrame:
    rows = load_jsonl(path)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    csv_path = path.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    print(title)
    print(f"JSONL: {path}")
    print(f"CSV  : {csv_path}")
    view = df.tail(tail)
    try:
        display(view)
    except NameError:
        print(view.to_string(index=False))
    return df


def find_raw_data(default_path: pathlib.Path) -> pathlib.Path:
    if default_path.exists():
        return default_path
    kaggle_input = pathlib.Path("/kaggle/input")
    candidates = sorted(kaggle_input.glob("**/train.csv.gz")) if kaggle_input.exists() else []
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        print("Multiple train.csv.gz candidates found:")
        for candidate in candidates:
            print(" -", candidate)
    raise FileNotFoundError(f"Raw data not found at {default_path}. Set RAW_DATA_PATH to the Kaggle file.")


print(f"Processed data : {PROCESSED_DIR}")
print(f"Outputs        : {OUTPUT_DIR}")
print(f"Config path    : {CONFIG_PATH}")
print(f"Fast debug     : {FAST_DEBUG}")


In [ ]:
# Cell 3 - Diffusion Pipeline Config
from src.utils.config import load_experiment_config


def deep_update(base: dict, override: dict) -> dict:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


config = load_experiment_config(
    base_path=str(REPO_DIR / "configs/base.yaml"),
    ablation_path=str(REPO_DIR / "configs/ablations/A8_multistep_diffusion.yaml"),
)
deep_update(
    config,
    {
        "data": {
            "max_seq_len": 256,
            "min_seq_len": 5,
            "event_type_col": "MCC",
            "timestamp_col": "TRDATETIME",
            "target_col": "target_flag",
            "numerical_cols": ["amount"],
            "categorical_cols": ["channel_type", "trx_category"],
            "group_col": "cl_id",
            "truncation_pretrain": "sliding_window",
            "truncation_eval": "last_events",
            "amount_transform": "robust_scaler",
            "amount_cols": ["amount"],
            "robust_scale_cols": ["amount"],
            "time_transform": "none",
            "min_vocab_count": 5,
            "train_ratio": 0.70,
            "val_ratio": 0.15,
            "test_ratio": 0.15,
            "use_time_features": False,
        },
        "pretraining": {"objective": "diffusion"},
        "diffusion": {
            "num_steps": DIFFUSION_STEPS,
            "timestep_sampling": "uniform",
            "beta_start": 0.0001,
            "beta_end": 0.02,
            "discrete_mask_fraction": 0.80,
        },
        "generation": {
            "enabled": False,
            "mode": "conditional_suffix",
            "suffix_len": GENERATION_SUFFIX_LEN,
            "prefix_min_ratio": 0.50,
            "prefix_max_ratio": 0.80,
            "sampler": "ddim_lite",
            "num_sampling_steps": GENERATION_SAMPLING_STEPS,
            "temperature_event_type": 1.0,
            "temperature_cat": 1.0,
            "top_k_event_type": 20,
        },
        "model": {
            "max_seq_len": 256,
            "use_profile_token": False,
            "client_embedding_dim": 512,
        },
        "pooling": {"type": "gated_attention"},
        "loss": {
            "lambda_event_type": 1.0,
            "lambda_time": 1.0,
            "lambda_num": 0.5,
            "lambda_cat": 0.5,
            "lambda_time_eps": 0.1,
            "lambda_num_eps": 0.1,
        },
        "loss_calibration": {"enabled": False},
        "training": {
            "task": "binary",
            "num_classes": 2,
            "main_metrics": ["roc_auc", "pr_auc", "macro_f1"],
            "batch_size": BATCH_SIZE,
            "num_epochs_pretrain": NUM_EPOCHS_PRETRAIN,
            "num_epochs_finetune": NUM_EPOCHS_FINETUNE,
            "mixed_precision": USE_AMP,
            "early_stopping_patience": 5,
            "log_every_n_steps": 50,
        },
    },
)

with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=False)

print(f"Saved diffusion config: {CONFIG_PATH}")
print(yaml.safe_dump({"pretraining": config["pretraining"], "diffusion": config["diffusion"], "loss": config["loss"]}, sort_keys=False))


In [ ]:
# Cell 4 - Data Loading & Preprocessing Artifacts
from src.data.preprocessing import EventPreprocessor
from src.data.splits import make_entity_splits


def load_raw_rosbank(path: pathlib.Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    ts_col = config["data"]["timestamp_col"]
    df[ts_col] = pd.to_datetime(df[ts_col], format="%d%b%y:%H:%M:%S", errors="coerce")
    if df[ts_col].isna().any():
        df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    if df[ts_col].isna().any():
        bad = int(df[ts_col].isna().sum())
        raise ValueError(f"Failed to parse {bad} timestamps in {ts_col}")
    return df


def write_raw_training_artifacts(raw_df: pd.DataFrame, splits: dict | None = None, prep: EventPreprocessor | None = None):
    data_cfg = config["data"]
    entity_col = data_cfg["group_col"]
    timestamp_col = data_cfg["timestamp_col"]
    target_col = data_cfg["target_col"]
    seed = int(config.get("seed", {}).get("global_seed", 42))

    df_sorted = raw_df.sort_values([entity_col, timestamp_col], kind="stable").reset_index(drop=True)
    helper = EventPreprocessor(config)
    df_sorted["time_delta"] = helper.compute_time_delta(df_sorted, entity_col, timestamp_col)

    if splits is None:
        splits = make_entity_splits(
            df_sorted,
            entity_col=entity_col,
            target_col=target_col,
            train_ratio=float(data_cfg.get("train_ratio", 0.70)),
            val_ratio=float(data_cfg.get("val_ratio", 0.15)),
            test_ratio=float(data_cfg.get("test_ratio", 0.15)),
            seed=seed,
            stratify=True,
        )
    if prep is None:
        prep = EventPreprocessor(config)
        prep.fit(df_sorted, splits["train"])

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df_sorted.to_parquet(PROCESSED_DIR / "events.parquet", index=False)
    with (PROCESSED_DIR / "preprocessor.pkl").open("wb") as f:
        pickle.dump(prep, f, protocol=pickle.HIGHEST_PROTOCOL)
    with (PROCESSED_DIR / "splits.json").open("w") as f:
        json.dump(splits, f, indent=2)
    return df_sorted, splits, prep


artifacts_exist = all((PROCESSED_DIR / name).exists() for name in ["events.parquet", "splits.json", "preprocessor.pkl"])
if artifacts_exist and not REBUILD_DATA:
    df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
    with (PROCESSED_DIR / "splits.json").open() as f:
        splits = json.load(f)
    with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
        prep = pickle.load(f)
    print("Loaded existing prepared artifacts")
else:
    RAW_DATA_PATH = find_raw_data(RAW_DATA_PATH)
    raw_df = load_raw_rosbank(RAW_DATA_PATH)
    RAW_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_PARQUET_PATH, index=False)
    print(f"Raw data: {raw_df.shape} from {RAW_DATA_PATH}")

    try:
        run_logged(
            [
                sys.executable,
                str(REPO_DIR / "scripts/prepare_data.py"),
                "--config",
                str(CONFIG_PATH),
                "--input",
                str(RAW_PARQUET_PATH),
                "--output-dir",
                str(PROCESSED_DIR),
            ],
            LOG_DIR / "prepare_data.log",
        )
        with (PROCESSED_DIR / "splits.json").open() as f:
            splits = json.load(f)
        with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
            prep = pickle.load(f)
        df_events, splits, prep = write_raw_training_artifacts(raw_df, splits=splits, prep=prep)
        print("Prepared artifacts with prepare_data.py; restored raw events.parquet for Dataset")
    except subprocess.CalledProcessError as exc:
        print(f"prepare_data.py failed with exit code {exc.returncode}; using inline artifact builder")
        df_events, splits, prep = write_raw_training_artifacts(raw_df)

print(f"Prepared events: {df_events.shape}")
print(f"Splits: train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")


In [ ]:
# Cell 5 - Prepared Artifact Inspection
df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
with (PROCESSED_DIR / "splits.json").open() as f:
    splits = json.load(f)
with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
    prep = pickle.load(f)

entity_col = config["data"]["group_col"]
target_col = config["data"]["target_col"]
event_type_col = config["data"]["event_type_col"]
seq_lens = df_events.groupby(entity_col).size()
class_counts = df_events.groupby(entity_col)[target_col].first().value_counts().sort_index()

print(f"Rows          : {len(df_events):,}")
print(f"Entities      : {df_events[entity_col].nunique():,}")
print(f"Sequence len  : min={seq_lens.min()}, p50={seq_lens.median():.0f}, max={seq_lens.max()}")
print(f"Class counts  : {dict(class_counts)}")
print(f"Event vocab   : {len(prep.vocab[event_type_col])}")
print(f"Cat vocabs    : {[len(prep.vocab[c]) for c in prep.categorical_cols]}")
print(f"Num cols      : {prep.numerical_cols}")
print(f"Cat cols      : {prep.categorical_cols}")


In [ ]:
# Cell 6 - Diffusion Smoke Test
from torch.utils.data import DataLoader

from src.corruption.diffusion import DiffusionCorruptionPipeline
from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import get_num_feature_dim
from src.models.dme_encoder import DMEEncoder
from src.training.losses import compute_diffusion_pretraining_loss


def to_device(value, target_device):
    if isinstance(value, torch.Tensor):
        return value.to(target_device)
    if isinstance(value, dict):
        return {k: to_device(v, target_device) for k, v in value.items()}
    if isinstance(value, list):
        return [to_device(v, target_device) for v in value]
    return value


def build_vocab_info(preprocessor, cfg: dict) -> dict:
    return {
        "event_type_vocab_size": len(preprocessor.vocab[preprocessor.event_type_col]),
        "cat_vocab_sizes": [len(preprocessor.vocab[col]) for col in preprocessor.categorical_cols],
        "num_num_features": get_num_feature_dim(preprocessor, cfg),
        "num_classes": int(cfg.get("training", {}).get("num_classes", 2)),
    }


smoke_ids = splits["train"][: min(len(splits["train"]), SMOKE_ENTITY_LIMIT)]
smoke_ds = EventSequenceDataset(df_events, smoke_ids, prep, config, mode="pretrain")
smoke_loader = DataLoader(smoke_ds, batch_size=SMOKE_BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
clean_batch = to_device(next(iter(smoke_loader)), device)

vocab_info = build_vocab_info(prep, config)
model = DMEEncoder(config, vocab_info).to(device)
pipe = DiffusionCorruptionPipeline(
    config=config,
    vocab_sizes={
        "event_type": vocab_info["event_type_vocab_size"],
        "cat_features": vocab_info["cat_vocab_sizes"],
    },
)

model.train()
corrupted_batch, targets, masks = pipe(clean_batch)
masks["attention_mask"] = corrupted_batch["attention_mask"]
outputs = model(corrupted_batch, mode="pretrain")
loss_dict = compute_diffusion_pretraining_loss(outputs, targets, masks, config)
loss_dict["total"].backward()

print(f"Smoke batch shape : event_type={tuple(clean_batch['event_type'].shape)}")
print(f"Timesteps sample  : {corrupted_batch['diffusion_t'][: min(8, len(corrupted_batch['diffusion_t']))].detach().cpu().tolist()}")
print("Diffusion losses  :", {k: round(float(v.detach().cpu()), 4) for k, v in loss_dict.items()})
print(f"Output keys       : {sorted(outputs.keys())}")

del model, pipe, clean_batch, corrupted_batch, outputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Cell 7 - Diffusion Pretraining
PRETRAIN_SUMMARY_PATH = PRETRAIN_OUTPUT_DIR / "metrics" / f"{CONFIG_PATH.stem}_pretrain_summary.json"

if RUN_DIFFUSION_PRETRAIN:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts/pretrain.py"),
        "--config",
        str(CONFIG_PATH),
        "--data-dir",
        str(PROCESSED_DIR),
        "--output-dir",
        str(PRETRAIN_OUTPUT_DIR),
    ]
    run_logged(cmd, LOG_DIR / "diffusion_pretrain.log")
else:
    print("RUN_DIFFUSION_PRETRAIN=False; expecting an existing summary")

assert PRETRAIN_SUMMARY_PATH.exists(), f"Missing pretrain summary: {PRETRAIN_SUMMARY_PATH}"
pretrain_summary = json.loads(PRETRAIN_SUMMARY_PATH.read_text())
DIFFUSION_CKPT_PATH = pathlib.Path(pretrain_summary["best_checkpoint"])
assert DIFFUSION_CKPT_PATH.exists(), f"Missing checkpoint: {DIFFUSION_CKPT_PATH}"

print(f"Diffusion pretrain summary: {PRETRAIN_SUMMARY_PATH}")
print(f"Diffusion checkpoint      : {DIFFUSION_CKPT_PATH}")
PRETRAIN_METRICS_JSONL = PRETRAIN_OUTPUT_DIR / "logs" / f"{CONFIG_PATH.stem}_pretrain" / "metrics.jsonl"
pretrain_metrics_df = show_metrics_tail(PRETRAIN_METRICS_JSONL, "Diffusion pretrain train/val metrics")


In [ ]:
# Cell 8 - Diffusion Validation Metrics
from src.training.pretrain import _batch_to_device


def evaluate_diffusion_validation(model, val_loader, pipeline, cfg, target_device) -> dict:
    model.eval()
    sums = {k: 0.0 for k in ["total", "event_type", "time_delta", "numerical", "categorical", "time_delta_eps", "numerical_eps"]}
    n_batches = 0
    correct_types = 0
    total_type_positions = 0
    time_abs_error = 0.0
    time_positions = 0
    time_eps_sq_error = 0.0
    time_eps_positions = 0

    with torch.no_grad():
        for clean_batch in val_loader:
            clean_batch = _batch_to_device(clean_batch, target_device)
            corrupted_batch, targets, masks = pipeline(clean_batch)
            masks["attention_mask"] = corrupted_batch["attention_mask"]
            outputs = model(corrupted_batch, mode="pretrain")
            loss_dict = compute_diffusion_pretraining_loss(outputs, targets, masks, cfg)
            for key in sums:
                sums[key] += float(loss_dict[key].detach().cpu())

            type_mask = masks["event_type"]
            if type_mask.any():
                preds = outputs["event_type_logits"][type_mask].argmax(dim=-1)
                trues = targets["event_type"][type_mask]
                correct_types += int((preds == trues).sum().item())
                total_type_positions += int(type_mask.sum().item())

            time_mask = masks["time_delta"]
            if time_mask.any():
                pred = outputs["time_delta_pred"].squeeze(-1)[time_mask]
                true = targets["time_delta"][time_mask]
                time_abs_error += float((pred - true).abs().sum().item())
                time_positions += int(time_mask.sum().item())

            eps_mask = masks["time_delta_eps"]
            if eps_mask.any():
                pred_eps = outputs["time_delta_eps_pred"].squeeze(-1)[eps_mask]
                true_eps = targets["time_delta_eps"][eps_mask]
                time_eps_sq_error += float(((pred_eps - true_eps) ** 2).sum().item())
                time_eps_positions += int(eps_mask.sum().item())

            n_batches += 1

    metrics = {f"loss_{k}": v / max(1, n_batches) for k, v in sums.items()}
    metrics["event_type_accuracy"] = correct_types / max(1, total_type_positions)
    metrics["time_delta_mae"] = time_abs_error / max(1, time_positions)
    metrics["time_delta_eps_mse"] = time_eps_sq_error / max(1, time_eps_positions)
    return metrics


batch_size = config["training"]["batch_size"]
val_loader = DataLoader(
    EventSequenceDataset(df_events, splits["val"], prep, config, mode="pretrain"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)
vocab_info = build_vocab_info(prep, config)
model = DMEEncoder(config, vocab_info)
ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
pipeline = DiffusionCorruptionPipeline(
    config=config,
    vocab_sizes={"event_type": vocab_info["event_type_vocab_size"], "cat_features": vocab_info["cat_vocab_sizes"]},
)

diffusion_val_metrics = evaluate_diffusion_validation(model, val_loader, pipeline, config, device)
DIFFUSION_VAL_METRICS_PATH = PRETRAIN_OUTPUT_DIR / "metrics" / "diffusion_validation_metrics.json"
DIFFUSION_VAL_METRICS_PATH.write_text(json.dumps(diffusion_val_metrics, indent=2))
print(f"Saved diffusion validation metrics: {DIFFUSION_VAL_METRICS_PATH}")
try:
    display(pd.DataFrame([diffusion_val_metrics]))
except NameError:
    print(diffusion_val_metrics)

del model, pipeline
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Cell 9 - Fine-tuning Frozen and Full Encoder
FROZEN_METRICS_PATH = FROZEN_OUTPUT_DIR / "metrics" / f"{CONFIG_PATH.stem}_finetune_metrics.json"
FULL_METRICS_PATH = FULL_OUTPUT_DIR / "metrics" / f"{CONFIG_PATH.stem}_finetune_metrics.json"

if RUN_FROZEN_FINETUNE:
    run_logged(
        [
            sys.executable,
            str(REPO_DIR / "scripts/finetune.py"),
            "--config",
            str(CONFIG_PATH),
            "--pretrained-checkpoint",
            str(DIFFUSION_CKPT_PATH),
            "--data-dir",
            str(PROCESSED_DIR),
            "--output-dir",
            str(FROZEN_OUTPUT_DIR),
            "--frozen-encoder",
        ],
        LOG_DIR / "finetune_frozen.log",
    )
else:
    print("RUN_FROZEN_FINETUNE=False; skipping frozen encoder fine-tune")

if RUN_FULL_FINETUNE:
    run_logged(
        [
            sys.executable,
            str(REPO_DIR / "scripts/finetune.py"),
            "--config",
            str(CONFIG_PATH),
            "--pretrained-checkpoint",
            str(DIFFUSION_CKPT_PATH),
            "--data-dir",
            str(PROCESSED_DIR),
            "--output-dir",
            str(FULL_OUTPUT_DIR),
        ],
        LOG_DIR / "finetune_full.log",
    )
else:
    print("RUN_FULL_FINETUNE=False; skipping full fine-tune")

rows = []
if FROZEN_METRICS_PATH.exists():
    rows.append({"pipeline": "frozen_encoder", **json.loads(FROZEN_METRICS_PATH.read_text()).get("test_metrics", {})})
if FULL_METRICS_PATH.exists():
    rows.append({"pipeline": "full_finetune", **json.loads(FULL_METRICS_PATH.read_text()).get("test_metrics", {})})

finetune_metrics_df = pd.DataFrame(rows)
FINETUNE_RESULTS_CSV = OUTPUT_DIR / "diffusion_finetune_metrics.csv"
finetune_metrics_df.to_csv(FINETUNE_RESULTS_CSV, index=False)
print(f"Saved fine-tune metrics: {FINETUNE_RESULTS_CSV}")
try:
    display(finetune_metrics_df)
except NameError:
    print(finetune_metrics_df.to_string(index=False))

FROZEN_TRAIN_METRICS_JSONL = FROZEN_OUTPUT_DIR / "logs" / f"{CONFIG_PATH.stem}_finetune" / "metrics.jsonl"
FULL_TRAIN_METRICS_JSONL = FULL_OUTPUT_DIR / "logs" / f"{CONFIG_PATH.stem}_finetune" / "metrics.jsonl"
if RUN_FROZEN_FINETUNE:
    frozen_train_metrics_df = show_metrics_tail(FROZEN_TRAIN_METRICS_JSONL, "Frozen encoder fine-tune train/val metrics")
if RUN_FULL_FINETUNE:
    full_train_metrics_df = show_metrics_tail(FULL_TRAIN_METRICS_JSONL, "Full fine-tune train/val metrics")


In [ ]:
# Cell 10 - Optional Low-Label Evaluation
if RUN_LOW_LABEL:
    run_logged(
        [
            sys.executable,
            str(REPO_DIR / "scripts/run_low_label.py"),
            "--config",
            str(CONFIG_PATH),
            "--pretrained-checkpoint",
            str(DIFFUSION_CKPT_PATH),
            "--data-dir",
            str(PROCESSED_DIR),
            "--output-dir",
            str(LOW_LABEL_OUTPUT_DIR),
        ],
        LOG_DIR / "low_label.log",
    )
    LOW_LABEL_AGG_CSV = LOW_LABEL_OUTPUT_DIR / "metrics" / "low_label_agg.csv"
    assert LOW_LABEL_AGG_CSV.exists(), f"Missing low-label aggregate: {LOW_LABEL_AGG_CSV}"
    low_label_df = pd.read_csv(LOW_LABEL_AGG_CSV)
    try:
        display(low_label_df)
    except NameError:
        print(low_label_df.to_string(index=False))
else:
    print("RUN_LOW_LABEL=False; set it to True for low-label diffusion evaluation")


In [ ]:
# Cell 11 - Event Suffix Generation
GENERATION_METRICS_PATH = GENERATION_OUTPUT_DIR / "generation_metrics.json"
GENERATED_EVENTS_PATH = GENERATION_OUTPUT_DIR / "generated_events.parquet"

if RUN_EVENT_GENERATION:
    run_logged(
        [
            sys.executable,
            str(REPO_DIR / "scripts/generate_events.py"),
            "--config",
            str(CONFIG_PATH),
            "--checkpoint",
            str(DIFFUSION_CKPT_PATH),
            "--data-dir",
            str(PROCESSED_DIR),
            "--output-dir",
            str(GENERATION_OUTPUT_DIR),
            "--split",
            "test",
            "--num-entities",
            str(GENERATION_ENTITY_LIMIT),
            "--num-samples",
            str(GENERATION_NUM_SAMPLES),
            "--suffix-len",
            str(GENERATION_SUFFIX_LEN),
        ],
        LOG_DIR / "generation.log",
    )
    generation_metrics = json.loads(GENERATION_METRICS_PATH.read_text())
    print(f"Generated events: {GENERATED_EVENTS_PATH}")
    try:
        display(pd.DataFrame([generation_metrics]))
    except NameError:
        print(generation_metrics)
else:
    generation_metrics = {}
    print("RUN_EVENT_GENERATION=False; skipping event suffix generation")


In [ ]:
# Cell 12 - Artifact Summary
summary_payload = {
    "config": str(CONFIG_PATH),
    "processed_data": str(PROCESSED_DIR),
    "pretrain_summary": str(PRETRAIN_SUMMARY_PATH),
    "diffusion_checkpoint": str(DIFFUSION_CKPT_PATH),
    "diffusion_validation_metrics": diffusion_val_metrics,
    "frozen_metrics": str(FROZEN_METRICS_PATH) if FROZEN_METRICS_PATH.exists() else None,
    "full_metrics": str(FULL_METRICS_PATH) if FULL_METRICS_PATH.exists() else None,
    "finetune_results_csv": str(FINETUNE_RESULTS_CSV),
    "generation_metrics": generation_metrics,
    "generated_events": str(GENERATED_EVENTS_PATH) if GENERATED_EVENTS_PATH.exists() else None,
    "logs": str(LOG_DIR),
}
RESULTS_JSON_PATH = OUTPUT_DIR / "diffusion_pipeline_results.json"
RESULTS_JSON_PATH.write_text(json.dumps(summary_payload, indent=2))

print("Diffusion pipeline artifacts")
for key, value in summary_payload.items():
    if isinstance(value, dict):
        print(f"{key:30s}: inline metrics")
    else:
        print(f"{key:30s}: {value}")
print(f"Summary JSON: {RESULTS_JSON_PATH}")
